In [0]:
%sql
-- PERGUNTA 1: Evolução anual de lançamentos com suporte ao Português-BR (Última Década)
SELECT 
    c.ano_lancamento,
    COUNT(f.id_jogo) AS total_lancamentos,
    SUM(CASE WHEN l.tem_texto_ptbr = true THEN 1 ELSE 0 END) AS qtd_com_texto_ptbr,
    SUM(CASE WHEN l.tem_audio_ptbr = true THEN 1 ELSE 0 END) AS qtd_com_dublagem_ptbr,
    ROUND(SUM(CASE WHEN l.tem_texto_ptbr = true THEN 1 ELSE 0 END) * 100.0 / COUNT(f.id_jogo), 2) AS pct_adesao_texto_ptbr,
    ROUND(SUM(CASE WHEN l.tem_audio_ptbr = true THEN 1 ELSE 0 END) * 100.0 / COUNT(f.id_jogo), 2) AS pct_adesao_dublagem_ptbr
FROM workspace.gold.fato_desempenho_jogos f
INNER JOIN workspace.gold.dim_calendario c 
    ON f.id_calendario = c.id_calendario
INNER JOIN workspace.gold.dim_localizacao_br l 
    ON f.id_localizacao = l.id_localizacao
WHERE c.ano_lancamento BETWEEN 2015 AND 2025
GROUP BY c.ano_lancamento
ORDER BY c.ano_lancamento ASC;

ano_lancamento,total_lancamentos,qtd_com_texto_ptbr,qtd_com_dublagem_ptbr,pct_adesao_texto_ptbr,pct_adesao_dublagem_ptbr
2015,2515,429,97,17.06,3.86
2016,4140,577,121,13.94,2.92
2017,5922,779,198,13.15,3.34
2018,7461,1024,276,13.72,3.70
2019,7242,1070,272,14.77,3.76
2020,8804,1392,346,15.81,3.93
2021,11067,1845,430,16.67,3.89
2022,12288,1930,497,15.71,4.04
2023,14596,2701,615,18.51,4.21
2024,20033,4091,974,20.42,4.86


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- PERGUNTA 2: Impacto do nível de localização PT-BR na aprovação, volume de avaliações e engajamento
SELECT 
    l.nivel_localizacao_br,
    COUNT(f.id_jogo) AS qtd_jogos_analisados,
    ROUND(AVG(f.preco_usd), 2) AS ticket_medio_usd,
    ROUND(AVG(f.taxa_aprovacao_pct), 2) AS media_taxa_aprovacao_pct,
    ROUND(AVG(f.total_reviews), 0) AS media_volume_reviews,
    ROUND(AVG(f.tempo_medio_jogo_min) / 60.0, 1) AS tempo_medio_jogado_horas,
    SUM(CASE WHEN d.faixa_proprietarios_estimada NOT IN ('0 - 0', '0 - 20000') THEN 1 ELSE 0 END) AS qtd_jogos_acima_20k_vendas,
    ROUND(
        SUM(CASE WHEN d.faixa_proprietarios_estimada NOT IN ('0 - 0', '0 - 20000') THEN 1 ELSE 0 END) * 100.0 / COUNT(f.id_jogo), 
        2
    ) AS pct_jogos_acima_20k_vendas
FROM workspace.gold.fato_desempenho_jogos f
INNER JOIN workspace.gold.dim_localizacao_br l 
    ON f.id_localizacao = l.id_localizacao
INNER JOIN workspace.gold.dim_jogo_detalhes d 
    ON f.id_jogo = d.id_jogo
WHERE f.total_reviews >= 10
GROUP BY l.nivel_localizacao_br
ORDER BY media_taxa_aprovacao_pct DESC;

nivel_localizacao_br,qtd_jogos_analisados,ticket_medio_usd,media_taxa_aprovacao_pct,media_volume_reviews,tempo_medio_jogado_horas,qtd_jogos_acima_20k_vendas,pct_jogos_acima_20k_vendas
Apenas Texto / Legenda PT-BR,9875,7.24,79.59,8344.0,7.7,4862,49.24
Dublagem Completa (Áudio + Texto),2616,5.58,79.28,10545.0,9.3,1293,49.43
Sem Suporte ao Português-BR,44171,4.74,76.57,878.0,6.8,16314,36.93


In [0]:
%sql
-- PERGUNTA 3: Penalidade na taxa de aprovação por gênero quando NÃO há suporte ao Português-BR
SELECT 
    cl.genero_principal,
    COUNT(f.id_jogo) AS total_jogos_genero,
    COUNT(CASE WHEN l.tem_texto_ptbr = true THEN 1 END) AS jogos_com_ptbr,
    COUNT(CASE WHEN l.tem_texto_ptbr = false THEN 1 END) AS jogos_sem_ptbr,
    ROUND(AVG(CASE WHEN l.tem_texto_ptbr = true THEN f.taxa_aprovacao_pct END), 2) AS aprovacao_com_ptbr_pct,
    ROUND(AVG(CASE WHEN l.tem_texto_ptbr = false THEN f.taxa_aprovacao_pct END), 2) AS aprovacao_sem_ptbr_pct,
    ROUND(
        COALESCE(AVG(CASE WHEN l.tem_texto_ptbr = true THEN f.taxa_aprovacao_pct END), 0) - 
        COALESCE(AVG(CASE WHEN l.tem_texto_ptbr = false THEN f.taxa_aprovacao_pct END), 0), 
        2
    ) AS diferenca_positiva_ptbr_pp
FROM workspace.gold.fato_desempenho_jogos f
INNER JOIN workspace.gold.dim_classificacao_jogo cl 
    ON f.id_classificacao = cl.id_classificacao
INNER JOIN workspace.gold.dim_localizacao_br l 
    ON f.id_localizacao = l.id_localizacao
WHERE f.total_reviews > 0
  AND cl.genero_principal <> 'Não Classificado'
GROUP BY cl.genero_principal
HAVING COUNT(f.id_jogo) >= 20
ORDER BY total_jogos_genero DESC
LIMIT 15;

genero_principal,total_jogos_genero,jogos_com_ptbr,jogos_sem_ptbr,aprovacao_com_ptbr_pct,aprovacao_sem_ptbr_pct,diferenca_positiva_ptbr_pp
Action,35148,7138,28010,77.79,74.76,3.02
Adventure,17486,2846,14640,79.48,77.33,2.16
Casual,15007,2836,12171,80.62,76.08,4.54
Indie,7981,1262,6719,78.38,75.1,3.29
Simulation,1792,343,1449,72.4,69.15,3.25
RPG,1326,136,1190,79.42,75.86,3.57
Strategy,1227,145,1082,78.14,75.44,2.69
Free To Play,509,53,456,70.37,73.22,-2.85
Racing,440,127,313,76.91,68.8,8.11
Violent,340,10,330,52.16,59.66,-7.5


In [0]:
%sql
-- PERGUNTA 4: Relação entre Faixa de Preço, Retenção e Aprovação (Comparando Com PT-BR vs Geral)
SELECT 
    cl.faixa_preco,
    COUNT(f.id_jogo) AS total_jogos_faixa,
    SUM(CASE WHEN l.tem_texto_ptbr = true THEN 1 ELSE 0 END) AS qtd_jogos_ptbr,
    ROUND(AVG(f.preco_usd), 2) AS preco_medio_faixa_usd,
    ROUND(AVG(CASE WHEN f.mediana_tempo_jogo_min > 0 THEN f.mediana_tempo_jogo_min END) / 60.0, 2) AS media_da_mediana_horas,
    ROUND(AVG(CASE WHEN f.tempo_medio_jogo_min > 0 THEN f.tempo_medio_jogo_min END) / 60.0, 2) AS tempo_medio_jogado_horas,
    ROUND(AVG(f.taxa_aprovacao_pct), 2) AS taxa_aprovacao_media_pct
FROM workspace.gold.fato_desempenho_jogos f
INNER JOIN workspace.gold.dim_classificacao_jogo cl 
    ON f.id_classificacao = cl.id_classificacao
INNER JOIN workspace.gold.dim_localizacao_br l 
    ON f.id_localizacao = l.id_localizacao
GROUP BY cl.faixa_preco
ORDER BY cl.faixa_preco ASC;

faixa_preco,total_jogos_faixa,qtd_jogos_ptbr,preco_medio_faixa_usd,media_da_mediana_horas,tempo_medio_jogado_horas,taxa_aprovacao_media_pct
1. Gratuito (Free-to-Play),26660,0,0.0,19.96,null,0.05
2. Até US$ 10 (Econômico),86900,2,3.68,5.22,null,0.03
3. US$ 10.01 a US$ 30 (Intermediário),11150,0,16.23,6.93,null,0.03
4. Acima de US$ 30 (Premium / AAA),1141,0,89.51,7.56,null,0.0


In [0]:
%sql
-- PERGUNTA 5: Desempenho de retenção e pico de jogadores simultâneos por Modalidade e Porte (Indie vs Grandes Estúdios)
SELECT 
    cl.modalidade_jogador,
    CASE WHEN cl.is_indie = true THEN 'Estúdio Indie' ELSE 'Estúdio Tradicional / AAA' END AS porte_estudio,
    COUNT(f.id_jogo) AS total_jogos,
    SUM(CASE WHEN l.tem_texto_ptbr = true THEN 1 ELSE 0 END) AS jogos_localizados_ptbr,
    ROUND(AVG(f.pico_jogadores_ccu), 1) AS media_pico_ccu,
    MAX(f.pico_jogadores_ccu) AS maior_pico_ccu_registrado,
    ROUND(AVG(CASE WHEN f.tempo_medio_jogo_min > 0 THEN f.tempo_medio_jogo_min END) / 60.0, 2) AS tempo_medio_jogado_horas,
    ROUND(AVG(f.taxa_aprovacao_pct), 2) AS media_aprovacao_pct
FROM workspace.gold.fato_desempenho_jogos f
INNER JOIN workspace.gold.dim_classificacao_jogo cl 
    ON f.id_classificacao = cl.id_classificacao
INNER JOIN workspace.gold.dim_localizacao_br l 
    ON f.id_localizacao = l.id_localizacao
GROUP BY 
    cl.modalidade_jogador, 
    CASE WHEN cl.is_indie = true THEN 'Estúdio Indie' ELSE 'Estúdio Tradicional / AAA' END
ORDER BY cl.modalidade_jogador, porte_estudio;

modalidade_jogador,porte_estudio,total_jogos,jogos_localizados_ptbr,media_pico_ccu,maior_pico_ccu_registrado,tempo_medio_jogado_horas,media_aprovacao_pct
Exclusivo Single-player,Estúdio Indie,69119,13189,8.1,83936,15.63,76.63
Exclusivo Single-player,Estúdio Tradicional / AAA,25726,3935,20.0,27880,14.0,74.84
Multiplayer / Co-op,Estúdio Indie,13599,3161,77.6,164168,10.68,75.96
Multiplayer / Co-op,Estúdio Tradicional / AAA,7282,1912,598.1,1013936,24.14,72.66
Outros / Não Especificado,Estúdio Indie,164,30,815.7,91184,63.84,73.26
Outros / Não Especificado,Estúdio Tradicional / AAA,9961,246,7.6,15279,73.4,70.53


In [0]:
%sql
SELECT 
    appid,
    name,
    release_date,
    price,
    supported_languages,
    full_audio_languages,
    reviews,
    positive,
    negative,
    categories,
    genres,
    tags
FROM workspace.bronze.steam_games_raw
LIMIT 5;

appid,name,release_date,price,supported_languages,full_audio_languages,reviews,positive,negative,categories,genres,tags
2539430,Black Dragon Mage Playtest,"Aug 1, 2023",0.0,null,[],[],0,0,null,null,null
496350,Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",5.24,"Springtime, April: when the cherry trees come into full bloom. The protagonist Yukinari Sanada has returned to his hometown in Kanagawa Prefecture, Kamakura City, for the first time in seven years, and is greeted by his older cousin Sakura Narumi (complete with maid outfit). He wanted to live in peace, but his life at the academy placed on the coastlands becomes bustling and brilliant while surrounded by a lineup of girls with booming personalities like Hotaru Amano, the sharp tongued, half-Japanese beauty, and Alice Kamishiro, the lazy witch who loves modern-science and mail-orders. Eventually, our protagonist is coaxed into joining the action committee for the academy's traditional beauty contest by his close friend. He interacts with the heroines participating in this contest who are cute but each have an idiocyncrasy or two. Fun events (with their own difficulties) pop up again and again. Your story with 'her' about smiles, peace, passion, and love, all while borrowing the power of a witch, is about to begin! 'Supipara - Alice the Magical Conductor.' is an interactive novel themed around 'dynamicism' depicted through a rallying all of the production techniques minori has accumulated to date. See expressive character portraits with moving eyes, mouths, even hair! It is quite immersive in many aspects. Also, 'Supipara' won't conclude in a single game. It will be released as several standalone episodes. Fans may freely pursue only the episodes that interest them if so desired. You could say 'it's a light novel series, and independent stories which develop their own unique personalities all in one.'",['English'],[],0,252,MangaGamer,"Single-player,Steam Trading Cards,Steam Cloud,Family Sharing",Adventure
1034400,Mystery Solitaire The Black Raven,"May 6, 2019",4.99,"Immerse yourself in the most beloved, mystical and entrancing world of Edgar Allan Poe's ''The Raven!'' This work reflected the image of the poet himself, his life and mysterious, gloomy and wonderful poetics. Explore 12 locations connected with Poe's life and work, and earn lots of cool trophies. Levels of varying complexity provide several hours of exciting gameplay every day, and special Golden Cards add an extra layer of fun. This game is for people who enjoy card games as spider solitaire, pyramid, hearts, klondike, free cell, crescent, golf, addiction, three peaks, yukon, forty thieves, scorpion, gaps, canfield, crazy quilt, algerian, montana, eight off, calculation, clock solitaire. But regardless of the type, solitaire is an age-old game that never loses its flavor! - Let your emotions run wild! - Solitaire card game – collect chains of cards! - Never a dull moment with super exciting levels! - Vivid graphics and a Pirate soundtrack make this Solitaire a win-win! - Get rid of cards faster with jokers, and increase the combo multiplier to earn more coins! - Earn 15 mystical items! - Original themed decks and 12 card back styles! - Let the fun take over!","['English', 'French', 'German', 'Russian']",[],0,21,8floor,"Single-player,Family Sharing",Casual
3292190,버튜버 파라노이아 - Vtuber Paranoia,"Oct 31, 2024",8.99,"synopsis 'Hello, I'm Hiyoro, a new YouTuber!' It's been three years since I've repeated such greetings. Hiyoro, who couldn't avoid Hakko's broadcast, fell into a lot of skepticism, self-destruction, and debt. Now, the broadcast that I turned on with the idea of the last broadcast, with an unknown algorithm, got a lot of response, and it ended up in a mess- Hiyoru, who suddenly received everyone's attention, became a popular YouTuber... Survive the controversy, red pills, and video donne! The survival of new YouTuber Hiyoru, 'Berber YouTuber Paranoia' is starting now! the number of people produc

In [0]:
# Substitua a leitura df_raw no Notebook 01 por esta configuração:
caminho_volume = "dbfs:/Volumes/workspace/bronze/arquivos_brutos/games.csv"
df_raw = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .option("multiLine", "true")
    .option("quote", "\"")
    .option("escape", "\"")
    .option("unescapedQuoteHandling", "STOP_AT_DELIMITER")
    .load(caminho_volume)
)

In [0]:
%sql
-- INSIGHT DE IP 1: Matriz de Oportunidade (Saturação vs. Taxa de Sucesso Comercial e Retenção em PT-BR)
SELECT 
    cl.genero_principal,
    cl.modalidade_jogador,
    COUNT(f.id_jogo) AS qtd_concorrentes_lancados,
    ROUND(AVG(f.taxa_aprovacao_pct), 2) AS aprovacao_media_pct,
    ROUND(AVG(CASE WHEN f.tempo_medio_jogo_min > 0 THEN f.tempo_medio_jogo_min END) / 60.0, 1) AS retencao_media_horas,
    -- Conta quantos jogos romperam a barreira de tração comercial (> 50 mil cópias estimadas)
    SUM(CASE 
        WHEN d.faixa_proprietarios_estimada NOT IN ('0 - 0', '0 - 20000', '20000 - 50000') 
        THEN 1 ELSE 0 
    END) AS qtd_jogos_sucesso_50k_mais,
    -- Taxa de Conversão de Sucesso Comercial (%) do nicho
    ROUND(
        SUM(CASE 
            WHEN d.faixa_proprietarios_estimada NOT IN ('0 - 0', '0 - 20000', '20000 - 50000') 
            THEN 1 ELSE 0 
        END) * 100.0 / COUNT(f.id_jogo), 
        2
    ) AS taxa_sucesso_comercial_pct
FROM workspace.gold.fato_desempenho_jogos f
INNER JOIN workspace.gold.dim_classificacao_jogo cl 
    ON f.id_classificacao = cl.id_classificacao
INNER JOIN workspace.gold.dim_localizacao_br l 
    ON f.id_localizacao = l.id_localizacao
INNER JOIN workspace.gold.dim_jogo_detalhes d 
    ON f.id_jogo = d.id_jogo
INNER JOIN workspace.gold.dim_calendario c 
    ON f.id_calendario = c.id_calendario
WHERE l.tem_texto_ptbr = true
  AND c.ano_lancamento >= 2020
  AND cl.is_indie = true -- Filtra o escopo de estúdios independentes/médios (remova se quiser incluir AAA)
  AND cl.genero_principal <> 'Não Classificado'
GROUP BY 
    cl.genero_principal,
    cl.modalidade_jogador
HAVING COUNT(f.id_jogo) >= 20 -- Garante amostra mínima estatística por nicho
ORDER BY taxa_sucesso_comercial_pct DESC, retencao_media_horas DESC;

genero_principal,modalidade_jogador,qtd_concorrentes_lancados,aprovacao_media_pct,retencao_media_horas,qtd_jogos_sucesso_50k_mais,taxa_sucesso_comercial_pct
Action,Multiplayer / Co-op,1530,79.02,11.5,353,23.07
Adventure,Multiplayer / Co-op,242,80.74,14.3,55,22.73
Indie,Multiplayer / Co-op,221,76.58,11.3,41,18.55
Adventure,Exclusivo Single-player,2204,81.75,6.2,291,13.20
Indie,Exclusivo Single-player,1312,79.22,11.2,169,12.88
Casual,Multiplayer / Co-op,375,80.25,30.8,44,11.73
Action,Exclusivo Single-player,4151,79.43,6.7,460,11.08
Casual,Exclusivo Single-player,3186,83.3,10.4,160,5.02


In [0]:
%sql
-- INSIGHT DE IP 2: Estratégia de Monetização, Faixa de Preço e Vida Útil via DLCs em Jogos Localizados PT-BR
SELECT 
    cl.faixa_preco,
    CASE 
        WHEN f.qtd_dlc = 0 THEN '1. Sem DLC (Jogo Fechado)'
        WHEN f.qtd_dlc BETWEEN 1 AND 3 THEN '2. 1 a 3 DLCs (Expansão Moderada)'
        ELSE '3. 4+ DLCs (IP como Plataforma / Live Content)'
    END AS estrategia_expansao_dlc,
    COUNT(f.id_jogo) AS total_jogos,
    ROUND(AVG(f.preco_usd), 2) AS preco_base_medio_usd,
    ROUND(AVG(f.taxa_aprovacao_pct), 2) AS aprovacao_media_pct,
    ROUND(AVG(f.total_reviews), 0) AS media_engajamento_reviews,
    ROUND(AVG(f.pico_jogadores_ccu), 1) AS media_pico_simultaneo_ccu,
    ROUND(AVG(CASE WHEN f.tempo_medio_jogo_min > 0 THEN f.tempo_medio_jogo_min END) / 60.0, 1) AS tempo_medio_vida_util_horas
FROM workspace.gold.fato_desempenho_jogos f
INNER JOIN workspace.gold.dim_classificacao_jogo cl 
    ON f.id_classificacao = cl.id_classificacao
INNER JOIN workspace.gold.dim_localizacao_br l 
    ON f.id_localizacao = l.id_localizacao
WHERE l.tem_texto_ptbr = true
  AND f.total_reviews >= 10 -- Filtra jogos que tiveram tração mínima real de público
GROUP BY 
    cl.faixa_preco,
    CASE 
        WHEN f.qtd_dlc = 0 THEN '1. Sem DLC (Jogo Fechado)'
        WHEN f.qtd_dlc BETWEEN 1 AND 3 THEN '2. 1 a 3 DLCs (Expansão Moderada)'
        ELSE '3. 4+ DLCs (IP como Plataforma / Live Content)'
    END
ORDER BY cl.faixa_preco ASC, estrategia_expansao_dlc ASC;

faixa_preco,estrategia_expansao_dlc,total_jogos,preco_base_medio_usd,aprovacao_media_pct,media_engajamento_reviews,media_pico_simultaneo_ccu,tempo_medio_vida_util_horas
1. Gratuito (Free-to-Play),1. Sem DLC (Jogo Fechado),636,0.0,73.75,16413.0,1202.4,30.6
1. Gratuito (Free-to-Play),2. 1 a 3 DLCs (Expansão Moderada),178,0.0,79.56,78996.0,10054.3,49.7
1. Gratuito (Free-to-Play),3. 4+ DLCs (IP como Plataforma / Live Content),116,0.0,76.46,75825.0,3404.8,42.5
2. Até US$ 10 (Econômico),1. Sem DLC (Jogo Fechado),6604,3.49,79.07,1765.0,32.0,9.2
2. Até US$ 10 (Econômico),2. 1 a 3 DLCs (Expansão Moderada),2612,4.29,82.93,6934.0,150.9,8.5
2. Até US$ 10 (Econômico),3. 4+ DLCs (IP como Plataforma / Live Content),503,5.37,81.41,35347.0,802.9,23.3
3. US$ 10.01 a US$ 30 (Intermediário),1. Sem DLC (Jogo Fechado),801,15.9,78.23,6379.0,236.0,14.7
3. US$ 10.01 a US$ 30 (Intermediário),2. 1 a 3 DLCs (Expansão Moderada),626,16.19,82.76,11829.0,731.2,16.0
3. US$ 10.01 a US$ 30 (Intermediário),3. 4+ DLCs (IP como Plataforma / Live Content),224,17.59,79.66,39550.0,2280.1,41.1
4. Acima de US$ 30 (Premium / AAA),1. Sem DLC (Jogo Fechado),128,152.12,51.44,3921.0,176.7,60.6
